# Data Cleaning: Secondary Victimization in School Violence Communication

This notebook turns the raw collection file `dataset.csv` into the **corpus layer** described in the project's *Dataset Description*.

**Dataset rules this notebook enforces**
- One row = one unique public communication. A text republished at another URL counts once.
- UTF-8 CSV, fully quoted, Vietnamese text in Unicode **NFC**.
- No empty records: every row has `title`, `full_text`, `source_url`, `source_type`.

**Corpus-layer fields produced:** `doc_id`, `source_type`, `source_name`, `source_url`, `publication_date` (YYYY-MM-DD), `publication_year`, `title`, `full_text`, `word_count`, `collector`.

**Outputs** (all in `outputs/`)
| File | Contents |
|---|---|
| `dataset_clean.csv` | The cleaned corpus |
| `excluded_rows.csv` | Rows dropped, with the reason. Fix them in **Manual fixes** (section 1) and re-run |
| `cleaning_log.csv` | Every automatic change to a value (row, field, old value, new value, rule) |
| `review_flags.csv` | Rows kept but worth a human check |
| `doc_id_registry.csv` | Permanent URL → `doc_id` mapping, so IDs never get renumbered |

The notebook is safe to re-run from the top. It never changes `dataset.csv`.

## 0. Setup

In [1]:
!pip3 install pandas underthesea

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-pac

kages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 'user = true' in your pip.conf
    file. Failure to do this can result in a broken Homebrew installation.
    
    Read more about this behavior here: <https://peps.python.org/pep-0668/>

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [2]:
# If a package is missing, run:  %pip install pandas underthesea
import re
import csv
import unicodedata
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

import pandas as pd

try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except ImportError:
    HAS_UNDERTHESEA = False
    print("WARNING: underthesea is not installed, so word_count falls back to syllable counts. "
          "Run `%pip install underthesea` and re-run the notebook.")

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. Configuration and manual fixes

Everything a person decides lives here. Edit these values and re-run the notebook; you do not need to touch the cleaning code below.

In [3]:
RAW_PATH = Path("dataset.csv")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

DOC_ID_PREFIX = "SV"            # doc_id format: SV0001, SV0002, ...
MIN_DATE = pd.Timestamp("2019-01-01")
COLLECTION_END_DATE = pd.Timestamp("2026-11-07")   # set to the official end of collection
MIN_WORDS = 100                 # minimum word_count (after word segmentation) to keep a text
KEEP_PHOTO_CAPTIONS = False     # proposal: captions are removed unless the team puts them in scope
REMOVE_EMOJI = True

VALID_SOURCE_TYPES = ["News", "School", "Government", "NGO"]
TEAM_MEMBERS = ["Bảo Anh", "Diễm Quí", "Linh Chi", "Trâm Anh", "Minh Ngọc"]

# Collector (formerly PIC) normalization.
# "Ngọc" is not on the team list. If it refers to a member, map it here, e.g. {"Ngọc": "Trâm Anh"}.
COLLECTOR_MAP = {
    "Ngọc": "Minh Ngọc",
}
# 20 School rows have no PIC. If the team knows who collected them, set the name here.
# While it is None, collector stays empty (null) for those rows and they are listed in review_flags.csv.
DEFAULT_COLLECTOR_FOR_MISSING = None   # team decision 2026-11-07: leave null until confirmed

# Per-row manual corrections, keyed by the ORIGINAL source_url (or by "row:<index>" when the URL is broken).
# Applied before any automatic cleaning. Use DD/MM/YYYY for dates, as in the raw file.
MANUAL_FIXES = {
    # Date typo "02/02/0202". Check the page and enter the real date:
    "https://tuson.bacninh.gov.vn/news/-/details/51508/chi-thi-cua-thu-tuong-chinh-phu-ve-ngan-chan-ay-lui-bao-luc-hoc-uong-89894817": {"publication_date": "02/02/2026"},
    #
    # Date "29/02/2022" does not exist (2022 is not a leap year):
    "http://thptnguyendu.edu.vn/tu-van-hoc-duong/bao-luc-hoc-duong-noi-am-anh-bao-gio-duoc-xoa-bo-267635": {"publication_date": "29/09/2022"},
    #
    # Missing date (Trường Tiểu học Vĩnh Hưng):
    "https://edu.viettel.vn/hni-hoangmai-thvinhhung/tin-tuc-su-kien/tin-cua-truong/bai-tuyen-truyen-phong-chong-bao-luc-hoc-duong.html": {"publication_date": ""},
    #
    # THCS Ngọc Thụy: the title was pasted into source_url. Enter the real URL:
    "row:215": {"source_url": "https://thcsngocthuy.hanoi.edu.vn/hoat-dong-chung/chao-co-tuan-4-tuyen-truyen-phong-chong-bao-luc-hoc-duong-ki-nang-bao-ve-ban-th/ct/6322/538524"},
}

In [4]:
# Canonical source names. Keys are names AFTER whitespace cleanup (see clean_source_name below);
# values are the single name used in the clean dataset.
SOURCE_NAME_CANONICAL = {
    # --- News outlets ---
    "Báo Thanh Niên": "Báo Thanh Niên", "Báo Thanh niên": "Báo Thanh Niên",
    "Báo Tuổi trẻ": "Báo Tuổi Trẻ", "Báo Tuổi Trẻ": "Báo Tuổi Trẻ", "Tuổi trẻ": "Báo Tuổi Trẻ",
    "Báo VnExpress": "VnExpress", "Báo VNExpress": "VnExpress", "Vietnamexpress": "VnExpress",
    "Vietnamnet": "VietNamNet", "Vietnam.net": "VietNamNet", "Báo Vietnamnet": "VietNamNet",
    "Báo Tiền Phong": "Báo Tiền Phong", "Tiền phong": "Báo Tiền Phong",
    "Hoa học trò": "Hoa Học Trò (Tiền Phong)",
    "Báo lao động": "Báo Lao Động", "Báo Lao động": "Báo Lao Động",
    "Báo Lao Động": "Báo Lao Động", "Lao Động": "Báo Lao Động",
    "Báo Người Lao Động": "Báo Người Lao Động",
    "Dân trí": "Báo Dân trí", "Báo Dân Trí": "Báo Dân trí", "Báo Dân trí": "Báo Dân trí",
    "Báo Dân": "Báo Dân trí",   # URL is dantri.com.vn
    "Báo Dân Việt": "Báo Dân Việt", "Dân việt": "Báo Dân Việt",
    "Báo Sài Gòn Giải Phóng": "Báo Sài Gòn Giải Phóng", "Sài Gòn Giải Phóng": "Báo Sài Gòn Giải Phóng",
    "Báo Giáo dục và Thời đại": "Báo Giáo dục và Thời đại", "Giá dục và Thời đại": "Báo Giáo dục và Thời đại",
    "Giáo dục Việt Nam": "Tạp chí Giáo dục Việt Nam", "Giá dục Việt Nam": "Tạp chí Giáo dục Việt Nam",
    "Tin tức thông tấn xã Việt Nam": "Tin tức TTXVN", "Tin tức Thông tấn xã Việt Nam": "Tin tức TTXVN",
    "Báo Chính Phủ": "Báo Điện tử Chính phủ", "Báo Điện tử Chính phủ": "Báo Điện tử Chính phủ",
    "Báo Điện Tử Chính phủ": "Báo Điện tử Chính phủ",
    "VTVonline": "VTV Online", "VTVOnline": "VTV Online",
    "VOV Tiếng nói Việt Nam": "VOV - Đài Tiếng nói Việt Nam",
    "Kênh 14": "Kenh14",
    # --- Schools ---
    "fpt school": "FPT Schools", "Fpt school": "FPT Schools", "FPT school": "FPT Schools",
    "THPT Nguyênc Du": "THPT Nguyễn Du",
    # A school press release (ISHCMC) republished by a newspaper; source_type is News, so name the outlet.
    # It is also flagged in review_flags.csv so the team can decide whether it belongs under School.
    "TCBC của trường được Báo giáo dục thời đại đăng lại": "Báo Giáo dục và Thời đại",
    # --- Government ---
    "UBND Tỉnh Thái Nguyên Cổng thông tin điện tử": "UBND Tỉnh Thái Nguyên",
    "Thành uỷ Hà Nội/ Ban Chấp hành Đảng bộ thành phố Hà Nội": "Thành ủy Hà Nội",
    "Cổng thông tin điện tử Tình Lạng sơn": "Cổng thông tin điện tử Tỉnh Lạng Sơn",
    "Cổng thông tin Điện tử Tỉnh Tuyên Quảng": "Cổng thông tin điện tử Tỉnh Tuyên Quang",
    "Cổng thông tin điện tử, công an tỉnh Điện biên": "Cổng thông tin điện tử Công an Tỉnh Điện Biên",
    "Công án Thành phố Đà Nẵng": "Công an Thành phố Đà Nẵng",
    "Phổ biến Giá dục Pháp luật Tỉnh Lai Châu": "Phổ biến giáo dục pháp luật Tỉnh Lai Châu",
}

# Source names that are wrong in the raw file (e.g. a headline pasted into source_name), keyed by URL domain.
SOURCE_NAME_BY_DOMAIN = {
    "bqlckcn.tuyenquang.gov.vn": "Ban Quản lý các Khu công nghiệp Tỉnh Tuyên Quang",
}

## 2. Load the raw data

Everything is read as text so nothing gets converted silently (for example, dates parsed the wrong way round).

In [5]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False, encoding="utf-8")
raw.index.name = "raw_row"          # 0-based row number in dataset.csv (header excluded)
print(raw.shape)
raw.head(3)

(260, 7)


,source_type,source_name,source_url,publication_date,title,full_text,PIC
raw_row,,,,,,,
0,NGO,UNICEF Viet Nam,https://www.unicef.org/vietnam/vi/tuyen-ngon-gioi-tre-cham-dut-bao-luc,24/06/2019,Tuyên ngôn giới trẻ #CHẤM_DỨT_Bạo_Lực,Những người trẻ tuổi lên tiếng muốn ngăn chặn bạo lực học đường.\n\nĐối với ...,Bảo Anh
1,NGO,UNICEF Viet Nam,https://www.unicef.org/vietnam/vi/th%C3%B4ng-c%C3%A1o-b%C3%A1o-ch%C3%AD/h%C6...,07/09/2018,Hơn 150 triệu thanh thiếu niên trên thế giới bị bạo lực học đường,Bạo lực học đường luôn là nỗi lo sợ của nhiều em nhỏ đang trong độ tuổi đến ...,Bảo Anh
2,NGO,UNICEF Viet Nam,https://www.unicef.org/vietnam/vi/th%C3%B4ng-c%C3%A1o-b%C3%A1o-ch%C3%AD/choi...,15/11/2018,Choi Siwon - Na Jaemin kêu gọi trẻ em Việt Nam cùng chống bạo lực học đường,"Hưởng ứng Ngày Trẻ em Thế giới, hai siêu sao Hàn Quốc Choi Siwon và Na Jaemi...",Bảo Anh


In [6]:
# Initial quality profile
profile = pd.DataFrame({
    "empty": (raw.apply(lambda s: s.str.strip()) == "").sum(),
    "has_leading/trailing_ws": (raw != raw.apply(lambda s: s.str.strip())).sum(),
    "has_line_break": raw.apply(lambda s: s.str.contains(r"[\r\n]")).sum(),
    "not_NFC": raw.apply(lambda s: (s != s.map(lambda x: unicodedata.normalize("NFC", x)))).sum(),
    "n_unique": raw.nunique(),
})
profile

,empty,has_leading/trailing_ws,has_line_break,not_NFC,n_unique
source_type,0,51,0,0,4
source_name,0,1,2,0,152
source_url,0,2,1,0,259
publication_date,1,0,0,0,211
title,7,11,8,1,242
full_text,0,44,260,8,256
PIC,20,0,0,0,6


In [7]:
print(raw["source_type"].map(repr).value_counts(), "\n")
print(raw["PIC"].map(repr).value_counts())

source_type
'News'          97
'Government'    82
'NGO '          51
'School'        30
Name: count, dtype: int64 

PIC
'Linh Chi'    106
'Bảo Anh'      75
'Trâm Anh'     33
'Diễm Quí'     25
''             20
'Ngọc'          1
Name: count, dtype: int64


## 3. Cleaning helpers

**Logging.** Each automatic change is recorded with `log_change`. Each dropped row is recorded with `exclude`, so every step can be traced and undone.

In [8]:
change_log = []     # rows: raw_row, field, old, new, rule
excluded = []       # rows: raw_row, reason, + original record
flags = []          # rows: raw_row, flag, detail

def log_change(idx, field, old, new, rule):
    if old != new:
        change_log.append({"raw_row": idx, "field": field, "old": old, "new": new, "rule": rule})

def flag(idx, name, detail=""):
    flags.append({"raw_row": idx, "flag": name, "detail": detail})

def apply_rule(df, field, func, rule):
    # Apply func to one column and log each value it changes
    new = df[field].map(func)
    for idx in df.index[df[field] != new]:
        log_change(idx, field, df.at[idx, field], new[idx], rule)
    df[field] = new
    return df

def exclude(df, mask, reason):
    # Move rows matching mask to the excluded list; return the remaining rows
    for idx, row in df[mask].iterrows():
        excluded.append({"raw_row": idx, "reason": reason, **raw.loc[idx].to_dict()})
    return df[~mask].copy()

In [9]:
ZERO_WIDTH = re.compile(r"[​‌‍⁠﻿­]")
EMOJI = re.compile(
    "[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF️‍⭐⭕⌚-⏿]"
)

def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s)

def clean_inline(s: str) -> str:
    # Single-line field: NFC, no zero-width chars, all whitespace (incl. line breaks) collapsed to one space
    s = ZERO_WIDTH.sub("", nfc(s)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", s).strip()

def clean_source_name(s: str) -> str:
    s = clean_inline(s)
    s = re.sub(r"\s*/\s*", "/ ", s) if "/" in s else s
    s = re.sub(r"^Công thông tin", "Cổng thông tin", s)          # missing hook on Cổng
    s = re.sub(r"điện từ", "điện tử", s, flags=re.I)              # typo từ -> tử
    s = re.sub(r"^(Cổng|Trang) thông tin (đ|Đ)iện (tử|Tử)", r"\1 thông tin điện tử", s, flags=re.I)
    s = re.sub(r"^cổng", "Cổng", s); s = re.sub(r"^trang", "Trang", s); s = re.sub(r"^trường", "Trường", s)
    if s.isupper() and len(s) > 12:                               # "CỔNG THÔNG TIN ĐIỆN TỬ TỈNH CÀ MAU" (not "VTV9")
        s = s.capitalize()
        s = re.sub(r"\b(tỉnh|xã|thành phố)\b(.*)$",
                   lambda m: m.group(1).capitalize() + m.group(2).title(), s)
    return SOURCE_NAME_CANONICAL.get(s, s)

def normalize_url(u: str) -> str:
    # Trim, drop tracking parameters (utm_*, fbclid, gclid) and the #fragment
    u = clean_inline(u)
    if not re.match(r"^https?://", u, flags=re.I):
        return u
    parts = urlsplit(u)
    query = [(k, v) for k, v in parse_qsl(parts.query, keep_blank_values=True)
             if not k.lower().startswith("utm_") and k.lower() not in {"fbclid", "gclid"}]
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), parts.path, urlencode(query), ""))

def url_key(u: str) -> str:
    # Key for duplicate detection: ignores http/https, www. and a trailing slash
    return re.sub(r"^https?://(www\.)?", "", u.lower()).rstrip("/")

In [10]:
SEPARATOR_LINE = re.compile(r"^\s*[_\-–—⸻=*~]{2,}\s*$")
ENGLISH_MARKER = re.compile(r"^\s*\[\s*English below\s*\]\s*", flags=re.I)
URL_ONLY_LINE = re.compile(r"^\s*(?:[\w\s]{0,20}:\s*)?(https?://\S+|www\.\S+|\S+\.(?:com|vn|org|cf|ly)/\S*)\s*$", flags=re.I)
BOILERPLATE_LINE = re.compile(
    r"^\s*(Xem thêm|Đọc thêm|Tin liên quan|Bài liên quan|TIN LIÊN QUAN|Video đang được quan tâm)\b.*$", flags=re.I
)
CAPTION_LINE = re.compile(r"^\s*\(?\s*(Ảnh|Hình ảnh|Nguồn ảnh|Video)\s*:.*$", flags=re.I)
VI_DIACRITIC = re.compile(r"[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", re.I)

def is_english_line(line: str, min_words: int = 8) -> bool:
    # A long line with no Vietnamese diacritics at all is an English paragraph
    # (a Vietnamese sentence of 8+ words without a single diacritic practically never occurs)
    words = re.findall(r"[A-Za-z']+", line)
    return len(words) >= min_words and not VI_DIACRITIC.search(line)

SLUG_LINE = re.compile(r"^\s*[a-z0-9]+(?:-[a-z0-9]+){2,}(?:\.(?:jpg|jpeg|png|webp|gif))?\s*$")   # image file names

TRAILING_LINK_SENTENCE = re.compile(r"(Để biết thêm chi tiết|Bạn muốn tìm hiểu thêm\?).*https?://\S+\s*$", flags=re.I)

def strip_english_translation(text: str) -> tuple[str, bool]:
    # Bilingual social posts: '[English below] <Vietnamese> __ <English>'. Keep the Vietnamese part only.
    if not ENGLISH_MARKER.match(text):
        return text, False
    text = ENGLISH_MARKER.sub("", text)
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if SEPARATOR_LINE.match(line):
            return "\n".join(lines[:i]), True
    return text, False      # marker but no separator: keep everything, flag for review

def clean_full_text(text: str):
    # Returns (cleaned_text, notes); notes lists the rules that fired
    notes = []
    t = ZERO_WIDTH.sub("", nfc(text)).replace("\xa0", " ").replace("\r\n", "\n").replace("\r", "\n")
    t, cut = strip_english_translation(t)
    if cut:
        notes.append("removed English translation")
    kept = []
    for line in t.split("\n"):
        if BOILERPLATE_LINE.match(line):
            notes.append("removed 'Xem thêm'/related-link line"); continue
        if URL_ONLY_LINE.match(line) or TRAILING_LINK_SENTENCE.search(line):
            notes.append("removed link line"); continue
        if not KEEP_PHOTO_CAPTIONS and CAPTION_LINE.match(line):
            notes.append("removed photo caption"); continue
        if SEPARATOR_LINE.match(line):
            continue
        if SLUG_LINE.match(line):
            notes.append("removed image file name"); continue
        if is_english_line(line):
            notes.append("removed English paragraph"); continue
        if REMOVE_EMOJI:
            line = EMOJI.sub("", line)
        line = re.sub(r"[ \t]+", " ", line).strip()
        kept.append(line)
    t = "\n".join(kept)
    t = re.sub(r"\n{2,}", "\n", t).strip()       # paragraphs separated by a single \n
    return t, sorted(set(notes))

def make_title_from_text(text: str, max_words: int = 15) -> str:
    # Fallback title for social posts without one: the first sentence, cut to max_words
    first = re.split(r"(?<=[.!?…])\s|\n", text.strip(), maxsplit=1)[0]
    words = first.strip("“”\"' ").split()
    return " ".join(words[:max_words]) + (" …" if len(words) > max_words else "")

In [11]:
PUNCT_ONLY = re.compile(r"^[\W_]+$")

def count_words(text: str) -> int:
    # Words after Vietnamese word segmentation (multi-syllable words such as 'học sinh' count once)
    if HAS_UNDERTHESEA:
        tokens = word_tokenize(text)
    else:
        tokens = text.split()
    return sum(1 for tok in tokens if not PUNCT_ONLY.match(tok))

print(word_tokenize("Bạo lực học đường đang là vấn đề nhức nhối.") if HAS_UNDERTHESEA else "fallback mode")

['Bạo lực', 'học đường', 'đang', 'là', 'vấn đề', 'nhức nhối', '.']


## 4. Apply manual fixes

In [12]:
df = raw.copy()
n_applied = 0
for key, fixes in MANUAL_FIXES.items():
    if key.startswith("row:"):
        idxs = [int(key[4:])]
    else:
        idxs = df.index[df["source_url"].str.strip() == key].tolist()
    if not idxs:
        print(f"WARNING: manual fix key not found: {key}")
    for idx in idxs:
        for field, value in fixes.items():
            log_change(idx, field, df.at[idx, field], value, "manual fix")
            df.at[idx, field] = value
        n_applied += 1
print(f"{n_applied} of {len(MANUAL_FIXES)} manual fix entries matched a row and were applied")

4 of 4 manual fix entries matched a row and were applied


## 5. Field-by-field cleaning

### 5.1 `source_type`
Trim, then map to exactly one of the four valid values, ignoring case.

In [13]:
type_lookup = {t.lower(): t for t in VALID_SOURCE_TYPES}
df = apply_rule(df, "source_type", lambda s: type_lookup.get(clean_inline(s).lower(), clean_inline(s)),
                "trim / canonical source_type")
df = exclude(df, ~df["source_type"].isin(VALID_SOURCE_TYPES), "invalid or missing source_type")
df["source_type"].value_counts()

source_type
News          97
Government    82
NGO           51
School        30
Name: count, dtype: int64

### 5.2 `source_url`
Trim it, remove line breaks and tracking parameters (`utm_*`, which come from copying links out of ChatGPT), then check that it starts with `http(s)://`.

In [14]:
df = apply_rule(df, "source_url", normalize_url, "trim / remove tracking params")
bad_url = ~df["source_url"].str.match(r"^https?://\S+$")
print(df.loc[bad_url, ["source_name", "source_url"]])
df = exclude(df, bad_url, "source_url missing or not a URL (fix in MANUAL_FIXES)")

Empty DataFrame
Columns: [source_name, source_url]
Index: []


### 5.3 `source_name`: one canonical name per organization

In [15]:
df = apply_rule(df, "source_name", clean_source_name, "canonical source_name")
domain = df["source_url"].map(lambda u: urlsplit(u).netloc.removeprefix("www."))
for idx in df.index[domain.isin(SOURCE_NAME_BY_DOMAIN)]:
    new = SOURCE_NAME_BY_DOMAIN[domain[idx]]
    log_change(idx, "source_name", df.at[idx, "source_name"], new, "source_name fixed from URL domain")
    df.at[idx, "source_name"] = new
for idx in df.index[raw.loc[df.index, "source_name"].str.contains("TCBC", na=False)]:
    flag(idx, "school_statement_republished_by_news", "consider source_type = School")
df = exclude(df, df["source_name"] == "", "missing source_name")

# Review the name list once per source type. Add new variants to SOURCE_NAME_CANONICAL.
for st in VALID_SOURCE_TYPES:
    names = df.loc[df["source_type"] == st, "source_name"].value_counts()
    print(f"\n=== {st}: {names.size} distinct names ===")
    print(names.to_string())


=== News: 24 distinct names ===
source_name
Báo Tuổi Trẻ                    14
VietNamNet                      10
Báo Thanh Niên                   9
Báo Dân trí                      8
Báo Lao Động                     8
VTV Online                       6
VnExpress                        6
Báo Tiền Phong                   6
Báo Sài Gòn Giải Phóng           4
Báo Giáo dục và Thời đại         4
VOV - Đài Tiếng nói Việt Nam     4
Báo Dân Việt                     3
Báo Phú Thọ                      2
Tạp chí Giáo dục Việt Nam        2
Báo Điện tử Chính phủ            2
VTV9                             1
Báo Người Lao Động               1
Huế ngày nay Online              1
Tin tức TTXVN                    1
Hoa Học Trò (Tiền Phong)         1
Báo Hải Phòng                    1
Kenh14                           1
Đời sống & Pháp luật             1
Báo Hà Tĩnh                      1

=== School: 27 distinct names ===
source_name
FPT Schools                                    3
THPT Nguyễn Du     

### 5.4 `publication_date`
Parse the date strictly as `DD/MM/YYYY` (also accepts `D/M/YYYY`). Impossible dates such as 29/02/2022 or year 0202 fail and go to `excluded_rows.csv`. Dates outside the study window are excluded too.

In [16]:
date_str = df["publication_date"].map(clean_inline)
parsed = pd.to_datetime(date_str, format="%d/%m/%Y", errors="coerce")

bad_date = parsed.isna() | (parsed.dt.year < 1990)    # e.g. year typed as 0202
print("Unparseable dates:\n", df.loc[bad_date, ["publication_date", "source_name"]], "\n")
df = exclude(df, bad_date, "publication_date missing or invalid (fix in MANUAL_FIXES)")
parsed = parsed[df.index]

out_of_window = (parsed < MIN_DATE) | (parsed > COLLECTION_END_DATE)
print("Outside study window:\n", df.loc[out_of_window, ["publication_date", "source_name", "title"]])
df = exclude(df, out_of_window,
             f"publication_date outside {MIN_DATE.date()} – {COLLECTION_END_DATE.date()}")
parsed = parsed[df.index]

iso = parsed.dt.strftime("%Y-%m-%d")
for idx in df.index:
    log_change(idx, "publication_date", df.at[idx, "publication_date"], iso[idx], "DD/MM/YYYY -> ISO")
df["publication_date"] = iso
df["publication_year"] = parsed.dt.year.astype(int)
df["publication_year"].value_counts().sort_index()

Unparseable dates:
         publication_date                source_name
raw_row                                            
236                       Trường Tiểu học Vĩnh Hưng 

Outside study window:
         publication_date      source_name                                                                        title
raw_row                                                                                                               
1             07/09/2018  UNICEF Viet Nam            Hơn 150 triệu thanh thiếu niên trên thế giới bị bạo lực học đường
2             15/11/2018  UNICEF Viet Nam  Choi Siwon - Na Jaemin kêu gọi trẻ em Việt Nam cùng chống bạo lực học đường
3             11/01/2017  UNICEF Viet Nam           Bảo vệ quyền của trẻ em trước thực trạng bạo lực, xâm hại tình dục
6             20/11/2018  UNICEF Viet Nam                 UNICEF Khuyến khích kỷ luật tích cực trong cách giáo dục trẻ
35            21/06/2017  UNICEF Viet Nam                       Hãy hành động để phòn

publication_year
2019    12
2020     9
2021     9
2022    35
2023    35
2024    42
2025    37
2026    74
Name: count, dtype: int64

### 5.5 `full_text`
- NFC normalization. Remove zero-width characters and non-breaking spaces. Line endings become `\n`.
- Bilingual social posts (`[English below] … __ English …`): keep only the Vietnamese part. Any other English-only paragraph (8+ words, no Vietnamese diacritics) is removed too.
- Remove "Xem thêm" / related-article lines, link-only lines, leftover image file names, and photo captions (unless `KEEP_PHOTO_CAPTIONS`). Remove emoji.
- Collapse spaces. Paragraphs are separated by a single `\n`.

In [17]:
text_notes = {}
new_text = {}
for idx, t in df["full_text"].items():
    cleaned, notes = clean_full_text(t)
    new_text[idx] = cleaned
    if notes:
        text_notes[idx] = notes
    if ENGLISH_MARKER.match(nfc(t)) and not {"removed English translation", "removed English paragraph"} & set(notes):
        flag(idx, "english_marker_without_separator", "English text may still be in full_text")
    # Log text changes as a summary (full texts are too long to store twice)
    if cleaned != t:
        change_log.append({"raw_row": idx, "field": "full_text",
                           "old": f"{len(t)} chars", "new": f"{len(cleaned)} chars",
                           "rule": "; ".join(notes) or "whitespace / NFC normalization"})
df["full_text"] = pd.Series(new_text)
pd.Series([n for ns in text_notes.values() for n in ns]).value_counts()

removed English translation             10
removed English paragraph                8
removed link line                        3
removed 'Xem thêm'/related-link line     2
removed photo caption                    1
removed image file name                  1
Name: count, dtype: int64

### 5.6 `title`
Keep it on one line and trim it. Seven UNICEF Facebook posts have no title. Their title is built from the first sentence of the post and flagged for review, so these rows are kept instead of dropped.

In [18]:
df = apply_rule(df, "title", clean_inline, "single line / trim / NFC")
missing_title = df["title"] == ""
for idx in df.index[missing_title]:
    new = make_title_from_text(df.at[idx, "full_text"])
    log_change(idx, "title", "", new, "title derived from first sentence (no title in source)")
    flag(idx, "title_derived", new)
    df.at[idx, "title"] = new
df.loc[missing_title, ["source_name", "title"]]

,source_name,title
raw_row,,
25,UNICEF Viet Nam,"Mình từng là 1 kẻ bắt nạt, sau một thời gian bị lãng quên, nạn …"
26,UNICEF Viet Nam,Bạn đã bao giờ bị bắt nạt trên mạng?
27,UNICEF Viet Nam,Giáo viên và trường học đóng vai trò quan trọng trong việc thúc đẩy sức …
28,UNICEF Viet Nam,"Chỉ đùa thôi mà!”, “Đừng có nghiêm túc quá!” Đôi khi, thật khó để biết …"
29,UNICEF Viet Nam,"Trong khuôn khổ chuyến đi thăm Đà Nẵng vừa qua, Đại diện đặc biệt của …"
33,UNICEF Viet Nam,32.5% các bạn học sinh nói rằng đã từng bị bắt nạt trên mạng - …
41,UNICEF Viet Nam,UNICEF Việt Nam giới thiệu hai đại biểu thanh niên đại diện Việt Nam tham …


### 5.7 `collector` (renamed from `PIC`)

In [19]:
df = df.rename(columns={"PIC": "collector"})

def clean_collector(s):
    s = clean_inline(s)
    s = COLLECTOR_MAP.get(s, s)
    if s == "" and DEFAULT_COLLECTOR_FOR_MISSING:
        s = DEFAULT_COLLECTOR_FOR_MISSING
    return s            # empty string = null (written as an empty cell)

df = apply_rule(df, "collector", clean_collector, "trim / map collector")
for idx in df.index[df["collector"] == ""]:
    flag(idx, "collector_missing", "PIC empty in raw data; left null until the team confirms")
for idx in df.index[(df["collector"] != "") & ~df["collector"].isin(TEAM_MEMBERS)]:
    flag(idx, "collector_not_team_member", df.at[idx, "collector"])
df["collector"].replace("", "(null)").value_counts()

collector
Linh Chi     106
Bảo Anh       69
Trâm Anh      32
Diễm Quí      25
(null)        20
Minh Ngọc      1
Name: count, dtype: int64

## 6. Word count and minimum length

In [20]:
df["word_count"] = df["full_text"].map(count_words)
syllables = df["full_text"].str.split().str.len()

short = df["word_count"] < MIN_WORDS
print(f"{short.sum()} texts under {MIN_WORDS} words:")
print(df.loc[short, ["source_name", "title", "word_count"]].assign(syllables=syllables[short]))
df = exclude(df, short, f"full_text under {MIN_WORDS} words after cleaning")
df["word_count"].describe()

12 texts under 100 words:
                    source_name                                                                            title  word_count  syllables
raw_row                                                                                                                                
24              UNICEF Viet Nam  Đức Phúc ám ảnh nhắc về quá khứ cũ - "Tôi từng là nạn nhân của bạo lực học đ...          52         65
25              UNICEF Viet Nam                 Mình từng là 1 kẻ bắt nạt, sau một thời gian bị lãng quên, nạn …          94        122
27              UNICEF Viet Nam        Giáo viên và trường học đóng vai trò quan trọng trong việc thúc đẩy sức …          76        106
28              UNICEF Viet Nam         Chỉ đùa thôi mà!”, “Đừng có nghiêm túc quá!” Đôi khi, thật khó để biết …          53         65
29              UNICEF Viet Nam          Trong khuôn khổ chuyến đi thăm Đà Nẵng vừa qua, Đại diện đặc biệt của …          68         99
30              UNICEF

count     241.000000
mean      598.987552
std       499.947960
min       110.000000
25%       299.000000
50%       473.000000
75%       738.000000
max      4449.000000
Name: word_count, dtype: float64

## 7. Duplicates

Proposal rule: *the same text republished at another URL counts once.*
1. **Same URL** (ignoring http/https, `www.`, trailing slash, tracking parameters).
2. **Same text** under different URLs. Comparison ignores case, punctuation and whitespace. Keep the **earliest** publication (the most likely original) and exclude the copies. Each exclusion records which `raw_row` was kept.
3. **Same title, different text.** These may be lightly edited reposts. They are flagged, not dropped.

In [21]:
df = df.sort_values(["publication_date"], kind="stable")

df["_url_key"] = df["source_url"].map(url_key)
df["_text_key"] = df["full_text"].map(lambda t: re.sub(r"[\W_]+", "", t.lower()))

def drop_dupes(df, key, reason):
    dup = df.duplicated(key, keep="first")
    kept_for = df.drop_duplicates(key).set_index(key).index
    first_idx = df.reset_index().drop_duplicates(key).set_index(key)["raw_row"]
    for idx in df.index[dup]:
        excluded.append({"raw_row": idx, "reason": f"{reason} (kept raw_row {first_idx[df.at[idx, key]]})",
                         **raw.loc[idx].to_dict()})
    print(f"{reason}: {dup.sum()} removed")
    return df[~dup].copy()

df = drop_dupes(df, "_url_key", "duplicate URL")
df = drop_dupes(df, "_text_key", "same text republished at another URL")

title_key = df["title"].str.lower().str.replace(r"[\W_]+", "", regex=True)
same_title = title_key.duplicated(keep=False)
for idx in df.index[same_title]:
    flag(idx, "same_title_different_text", df.at[idx, "title"])
print("Same title, different text (flagged):")
print(df.loc[same_title, ["source_name", "publication_date", "title", "word_count"]].sort_values("title"))
df = df.drop(columns=["_url_key", "_text_key"])

duplicate URL: 1 removed
same text republished at another URL: 6 removed
Same title, different text (flagged):
                                               source_name publication_date                                                                           title  word_count
raw_row                                                                                                                                                                
157              UBND Tỉnh Cà Mau - Sở Giáo dục và Đào tạo       2026-05-31  Bộ Giáo dục và Đào tạo yêu cầu tăng cường ngăn chặn, đẩy lùi bạo lực học đường         429
152                  Cổng thông tin điện tử Tỉnh Ninh Bình       2026-06-01  Bộ Giáo dục và Đào tạo yêu cầu tăng cường ngăn chặn, đẩy lùi bạo lực học đường         500
171                      Cổng thông tin điện tử Bộ Công an       2026-02-02       Ngăn chặn, đẩy lùi bạo lực học đường, bảo đảm an ninh, an toàn trường học         898
177                                  UBND Tỉnh Th

## 8. Scope check: language

The scope covers Vietnamese-language texts only. A text with few Vietnamese diacritics is probably mostly English, so it is flagged for review.

In [22]:
VI_CHARS = re.compile(r"[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", re.I)

def vi_ratio(text):
    letters = re.findall(r"[^\W\d_]", text)
    return len(VI_CHARS.findall(text)) / max(len(letters), 1)

ratio = df["full_text"].map(vi_ratio)
for idx in df.index[ratio < 0.10]:
    flag(idx, "possibly_not_vietnamese", f"diacritic ratio {ratio[idx]:.2f}")
ratio.describe()

count    234.000000
mean       0.288484
std        0.012298
min        0.236745
25%        0.280738
50%        0.289710
75%        0.296433
max        0.317144
Name: full_text, dtype: float64

## 9. Stable `doc_id`

IDs are stored in `outputs/doc_id_registry.csv`, keyed by URL. On a re-run, existing documents keep their ID and new documents get the next free number. IDs are never reused or renumbered, even after rows are dropped.

In [23]:
REGISTRY_PATH = OUT_DIR / "doc_id_registry.csv"
if REGISTRY_PATH.exists():
    registry = pd.read_csv(REGISTRY_PATH, dtype=str)
else:
    registry = pd.DataFrame(columns=["url_key", "doc_id"])

id_map = dict(zip(registry["url_key"], registry["doc_id"]))
next_num = max([int(d[len(DOC_ID_PREFIX):]) for d in id_map.values()], default=0) + 1

# Assign new IDs in raw-file order, so the first run gives SV0001 to the first kept row
df = df.sort_index()
keys = df["source_url"].map(url_key)
for k in keys:
    if k not in id_map:
        id_map[k] = f"{DOC_ID_PREFIX}{next_num:04d}"
        next_num += 1
df["doc_id"] = keys.map(id_map)

registry = pd.DataFrame({"url_key": list(id_map), "doc_id": list(id_map.values())})
registry.to_csv(REGISTRY_PATH, index=False, encoding="utf-8")
print(f"{len(registry)} IDs in registry; next new ID: {DOC_ID_PREFIX}{next_num:04d}")

234 IDs in registry; next new ID: SV0235


## 10. Validation against the corpus-layer specification

In [24]:
COLUMNS = ["doc_id", "source_type", "source_name", "source_url", "publication_date",
           "publication_year", "title", "full_text", "word_count", "collector"]
clean = df[COLUMNS].sort_values("doc_id").reset_index()

checks = {
    "doc_id unique, not null, format SV0000":
        clean["doc_id"].is_unique and clean["doc_id"].str.fullmatch(rf"{DOC_ID_PREFIX}\d{{4,}}").all(),
    "source_type in the 4 valid values": clean["source_type"].isin(VALID_SOURCE_TYPES).all(),
    "source_name not null": (clean["source_name"] != "").all(),
    "source_url unique, http(s), no whitespace":
        clean["source_url"].is_unique and clean["source_url"].str.fullmatch(r"https?://\S+").all(),
    "publication_date ISO and within window":
        clean["publication_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
        and pd.to_datetime(clean["publication_date"]).between(MIN_DATE, COLLECTION_END_DATE).all(),
    "publication_year matches date":
        (pd.to_datetime(clean["publication_date"]).dt.year == clean["publication_year"]).all(),
    "title not null, single line":
        (clean["title"] != "").all() and not clean["title"].str.contains(r"[\r\n]").any(),
    "full_text not null, no \\r, no blank lines":
        (clean["full_text"] != "").all() and not clean["full_text"].str.contains(r"\r|\n\n").any(),
    f"word_count >= {MIN_WORDS}": (clean["word_count"] >= MIN_WORDS).all(),
    "all text fields NFC": all(
        (clean[c] == clean[c].map(nfc)).all() for c in ["source_name", "title", "full_text", "collector"]),
    "no duplicate texts": not clean["full_text"].str.lower().str.replace(r"[\W_]+", "", regex=True).duplicated().any(),
}
report = pd.Series(checks, name="passed")
display(report.to_frame())
assert report.all(), "Some validation checks failed. See the table above."

# Known open item, reported but not blocking: the spec requires collector to be non-null
n_null_collector = (clean["collector"] == "").sum()
if n_null_collector:
    print(f"OPEN ITEM: {n_null_collector} rows have a null collector "
          "(set DEFAULT_COLLECTOR_FOR_MISSING or per-row MANUAL_FIXES once confirmed)")

,passed
"doc_id unique, not null, format SV0000",True
source_type in the 4 valid values,True
source_name not null,True
"source_url unique, http(s), no whitespace",True
publication_date ISO and within window,True
publication_year matches date,True
"title not null, single line",True
"full_text not null, no \r, no blank lines",True
word_count >= 100,True
all text fields NFC,True


OPEN ITEM: 19 rows have a null collector (set DEFAULT_COLLECTOR_FOR_MISSING or per-row MANUAL_FIXES once confirmed)


## 11. Save outputs

In [25]:
clean[COLUMNS].to_csv(OUT_DIR / "dataset_clean.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

excluded_df = pd.DataFrame(excluded).sort_values("raw_row")
excluded_df.to_csv(OUT_DIR / "excluded_rows.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

log_df = pd.DataFrame(change_log).sort_values(["raw_row", "field"], kind="stable")
log_df.to_csv(OUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

# Attach doc_id to flags for kept rows only
row_to_id = dict(zip(clean["raw_row"], clean["doc_id"]))
flags_df = pd.DataFrame(flags, columns=["raw_row", "flag", "detail"])
flags_df = flags_df[flags_df["raw_row"].isin(row_to_id)].assign(doc_id=lambda d: d["raw_row"].map(row_to_id))
flags_df = flags_df[["doc_id", "raw_row", "flag", "detail"]].drop_duplicates().sort_values("doc_id")
flags_df.to_csv(OUT_DIR / "review_flags.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

# Round-trip check: the saved file reads back identically
back = pd.read_csv(OUT_DIR / "dataset_clean.csv", dtype={"publication_year": int, "word_count": int},
                   keep_default_na=False)
assert back.equals(clean[COLUMNS].reset_index(drop=True)), "CSV round-trip mismatch"
print("Saved to", OUT_DIR.resolve())

Saved to /Users/Admin/Documents/GitHub Repositories/school-violence/outputs


## 12. Cleaning summary

In [26]:
print(f"Raw rows:      {len(raw)}")
print(f"Excluded:      {len(excluded_df)}")
print(f"Clean rows:    {len(clean)}\n")
print("Exclusions by reason:")
print(excluded_df["reason"].str.replace(r" \(kept raw_row \d+\)", "", regex=True).value_counts().to_string(), "\n")
print("Rows flagged for review:")
print(flags_df["flag"].value_counts().to_string(), "\n")
print("Automatic changes by field:")
print(log_df["field"].value_counts().to_string())

Raw rows:      260
Excluded:      26
Clean rows:    234

Exclusions by reason:
reason
full_text under 100 words after cleaning                     12
publication_date outside 2019-01-01 – 2026-11-07              6
same text republished at another URL                          6
duplicate URL                                                 1
publication_date missing or invalid (fix in MANUAL_FIXES)     1 

Rows flagged for review:
flag
collector_missing                       19
same_title_different_text                8
title_derived                            2
school_statement_republished_by_news     1 

Automatic changes by field:
field
publication_date    255
full_text           250
source_name          92
source_type          51
title                20
source_url           15
collector             1


In [27]:
# Quick look at group sizes for RQ2 (full descriptive analysis belongs in Phase 1 notebook)
summary = clean.groupby("source_type").agg(
    n_texts=("doc_id", "size"),
    n_outlets=("source_name", "nunique"),
    median_words=("word_count", "median"),
    first_date=("publication_date", "min"),
    last_date=("publication_date", "max"),
)
summary["share_%"] = (100 * summary["n_texts"] / summary["n_texts"].sum()).round(1)
summary

,n_texts,n_outlets,median_words,first_date,last_date,share_%
source_type,,,,,,
Government,76,67,523.0,2019-04-09,2026-08-29,32.5
NGO,35,4,668.0,2019-01-10,2026-09-04,15.0
News,95,24,352.0,2022-03-27,2026-09-23,40.6
School,28,25,657.0,2020-10-26,2026-09-23,12.0


In [28]:
pd.crosstab(clean["collector"].replace("", "(null)"), clean["source_type"], margins=True)

source_type,Government,NGO,News,School,All
collector,,,,,
(null),0,0,0,19,19
Bảo Anh,0,34,24,1,59
Diễm Quí,0,0,24,0,24
Linh Chi,71,0,29,0,100
Minh Ngọc,0,0,1,0,1
Trâm Anh,5,1,17,8,31
All,76,35,95,28,234


## 13. Open items for the team

These need a human decision. Each one is also listed in `excluded_rows.csv` or `review_flags.csv`.
1. **Fix the invalid dates** (02/02/0202, 29/02/2022, and one missing date) and the **missing URL** for THCS Ngọc Thụy in `MANUAL_FIXES`.
2. **Who collected the 20 School rows** with no PIC? They are left null for now (flag `collector_missing`). Set `DEFAULT_COLLECTOR_FOR_MISSING`, or add per-row fixes, once confirmed.
3. **Who is "Ngọc"?** If Ngọc is a team member under another name, map it in `COLLECTOR_MAP`.
4. **Texts published before 2019** are out of the study window and excluded. Change `MIN_DATE` if the team wants to keep them.
5. **Short social posts** (under 100 words after removing the English translation) are excluded. Decide whether campaign posts get a lower threshold.
6. **Group sizes.** The proposal targets about 100 documents per source type. The summary above shows how far each group is from that target.